# 01 – Đọc dữ liệu, thống kê sơ bộ và làm sạch

Notebook chỉ gọi hàm trong `src/` (issue #3, #4).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_raw, summarize, label_distribution
from src.preprocess import clean, negative_bill_ratio

pd.set_option('display.width', 140)

## 1. Đọc dữ liệu gốc

In [2]:
df = load_raw()
print(df.shape)
df.head()

(30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Dữ liệu có 30000 dòng × 24 cột (23 thuộc tính + nhãn `default`), đã đổi `PAY_0` thành `PAY_1` và bỏ cột `ID`.

## 2. Thống kê sơ bộ

In [3]:
summarize(df)

,dtype,n_unique,n_missing,min,max
LIMIT_BAL,int64,81,0,10000,1000000
SEX,int64,2,0,1,2
EDUCATION,int64,7,0,0,6
MARRIAGE,int64,4,0,0,3
AGE,int64,56,0,21,79
PAY_1,int64,11,0,-2,8
PAY_2,int64,11,0,-2,8
PAY_3,int64,11,0,-2,8
PAY_4,int64,11,0,-2,8
PAY_5,int64,10,0,-2,8


Không có giá trị thiếu ở cột nào, mọi cột đều là số nguyên. `EDUCATION` có giá trị 0–6 và `MARRIAGE` có giá trị 0–3 trong khi mô tả gốc chỉ định nghĩa 1–4 và 1–3, nên cần gộp ở bước làm sạch. Các cột `BILL_AMT*` có giá trị âm (sao kê âm).

## 3. Phân phối nhãn

In [4]:
label_distribution(df)

,count,ratio
default,,
0,23364,0.7788
1,6636,0.2212


Lớp vỡ nợ (1) chiếm khoảng 22,1% (6636/30000), lớp không vỡ nợ chiếm 77,9%: dữ liệu mất cân bằng vừa phải, nên cần chia phân tầng và dùng PR-AUC cùng các phương án tăng cường dữ liệu.

## 4. Làm sạch (bảng 2.2)

Gộp `EDUCATION` {0,5,6} → 4, `MARRIAGE` 0 → 3, rồi loại bản ghi trùng hoàn toàn (loại trùng sau khi gộp nhãn). Hàm trả về bản sao, không sửa `df`.

In [5]:
df_before = df.copy()
df_clean = clean(df)
print(df.shape, '->', df_clean.shape)
assert df.equals(df_before)  # df đầu vào không bị sửa
print(sorted(df_clean['EDUCATION'].unique()), sorted(df_clean['MARRIAGE'].unique()))

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)
(30000, 24) -> (29965, 24)
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)] [np.int64(1), np.int64(2), np.int64(3)]


Đã gộp 14 + 280 + 51 = 345 mẫu `EDUCATION` và 54 mẫu `MARRIAGE`, loại 35 dòng trùng hoàn toàn (còn 29965 dòng). Sau làm sạch `EDUCATION` ∈ {1,2,3,4} và `MARRIAGE` ∈ {1,2,3}.

## 5. Kiểm tra sao kê âm

Tỉ lệ dòng có `BILL_AMT_i < 0` theo từng cột và tỉ lệ dòng có ít nhất một sao kê âm (`any_of_6`), tính trên dữ liệu đã làm sạch.

In [6]:
negative_bill_ratio(df_clean)

,count,ratio
BILL_AMT1,590,0.019690
BILL_AMT2,669,0.022326
BILL_AMT3,655,0.021859
BILL_AMT4,675,0.022526
BILL_AMT5,655,0.021859
BILL_AMT6,688,0.022960
any_of_6,1930,0.064408


Mỗi cột có khoảng 2,0–2,3% dòng có sao kê âm, và 6,4% dòng (1930 dòng) có ít nhất một sao kê âm trong 6 tháng. Tỉ lệ nhỏ nên giữ nguyên các giá trị này, dùng `signed_log` ở bước mã hóa để xử lý dấu.